# 02 — Cluster Discovery: GMM K-Sweep

Discover the optimal number of behavioral syllables via GMM (K=10–80) on
independent 3000-frame chunks (1800-frame gaps).

### Recording structure
Each mouse: 3 experimental stages × 2 ages = 6 recordings.
Video name: `animal_id / exp_stage / age / session`.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from scipy.stats import entropy
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

sns.set_context("notebook", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120
print("Libraries loaded.")

## 1. Configuration

In [ ]:
from pathlib import Path

def find_base_dir(anchor="data"):
    """Walk upward from CWD until we find a child directory named `anchor`."""
    cwd = Path.cwd()
    for parent in [cwd] + list(cwd.parents):
        if (parent / anchor).is_dir():
            return parent
    raise FileNotFoundError(
        f"Could not find a '{anchor}/' directory in any parent of {cwd}."
    )

BASE_DIR = find_base_dir("data")
PROJECT_ROOT = Path("/scratch/michal/projects/dvc_ofd_2025")
EMB_H5_PATH = BASE_DIR / "data/embeddings/ofd_tailhip_20260226-205043.h5"
META_TSV_PATH = PROJECT_ROOT / "data/raw/openfield_ORT/hdp_meta.tsv"
# --- Independence parameters ---
# hBehaveMAE receptive field = 900 frames.
# 3000-frame chunks with 1800-frame gaps → zero information leakage.
STAGE = "stage2"            # 192D
CHUNK_FRAMES = 3000
GAP_FRAMES = 1800
STRIDE = CHUNK_FRAMES + GAP_FRAMES
# --- Filtering ---
HYBRID_STRAIN = "B6CAST-129SPWK-F2"
MIN_CHUNKS_PER_STRAIN = 20
RANDOM_STATE = 42

K_VALUES = list(range(10, 81, 5))
N_INIT = 3
MAX_ITER = 300
COVARIANCE_TYPE = "full"
MAX_FIT_FRAMES = 500_000

print(f"BASE_DIR: {BASE_DIR}")
print(f"K sweep: {K_VALUES[0]}→{K_VALUES[-1]} ({len(K_VALUES)} values), cov={COVARIANCE_TYPE}")

## 2. Load Metadata

In [ ]:
meta_df = pd.read_csv(META_TSV_PATH, sep=r'\s+')
meta_df.columns = meta_df.columns.str.replace('"', '').str.strip()
meta_df['animal_id'] = meta_df['animal_id'].astype(str).str.replace('"', '').str.strip()

strain_lookup = dict(zip(meta_df['animal_id'], meta_df['strain']))
print(f"Metadata loaded: {len(meta_df)} animals, {meta_df['strain'].nunique()} unique strains")

## 3. Extract Independent Chunks

In [ ]:
def extract_independent_chunks(embeddings, chunk_frames=3000, gap_frames=1800):
    """Slice (T, D) into independent chunks separated by gaps.

    If the video is shorter than chunk_frames, the entire video is
    returned as a single chunk (graceful fallback so no video is
    silently dropped when chunk_frames is set very large).
    """
    T = embeddings.shape[0]

    # Fallback: video shorter than one chunk → use the whole video
    if T < chunk_frames:
        return [embeddings]

    stride = chunk_frames + gap_frames
    chunks = []
    start = 0
    while start + chunk_frames <= T:
        chunks.append(embeddings[start : start + chunk_frames])
        start += stride
    return chunks
all_chunks = []
chunk_meta_rows = []

with h5py.File(EMB_H5_PATH, 'r') as f:
    video_names = sorted(f.keys())
    print(f"Videos in H5 file: {len(video_names)}")

    for vid in video_names:
        if STAGE not in f[vid]:
            continue

        emb = f[vid][STAGE][:]         # (T, 192)

        # Parse: HDP-013893_ACQ_Old_4 → animal_id, exp_stage, age
        parts = vid.split('_')
        if len(parts) < 4:
            continue
        animal_id = parts[0]
        exp_stage = parts[1]           # HAB, ACQ, etc.
        age       = parts[2]           # Adult, Old

        chunks = extract_independent_chunks(emb, CHUNK_FRAMES, GAP_FRAMES)

        for ci, chunk in enumerate(chunks):
            all_chunks.append(chunk)
            chunk_meta_rows.append({
                "pos": len(all_chunks) - 1,
                "video_name": vid,
                "animal_id": animal_id,
                "exp_stage": exp_stage,
                "age": age,
                "chunk_idx": ci,
            })

chunk_df = pd.DataFrame(chunk_meta_rows)
print(f"\nExtracted {len(all_chunks)} independent chunks")
print(f"  Videos:  {chunk_df['video_name'].nunique()}")
print(f"  Animals: {chunk_df['animal_id'].nunique()}")
print(f"  Exp stages: {sorted(chunk_df['exp_stage'].unique())}")
print(f"  Ages:       {sorted(chunk_df['age'].unique())}")
print(f"\nRecordings per animal (videos): "
      f"{chunk_df.groupby('animal_id')['video_name'].nunique().describe()[['mean','min','max']].to_dict()}")

## 4. Data Filtering

In [ ]:
print("=" * 60)
print("Applying Strict Data Filtering")
print("=" * 60)

# Step 1: Merge with metadata
master_df = pd.merge(chunk_df, meta_df, on='animal_id', how='inner')
print(f"After metadata merge: {len(master_df)} chunks")

if master_df.empty:
    raise RuntimeError(
        f"Merge produced 0 rows! Check animal_id formats.\n"
        f"  chunk IDs: {chunk_df['animal_id'].unique()[:3]}\n"
        f"  meta IDs:  {meta_df['animal_id'].unique()[:3]}"
    )

# Step 2: Drop missing strain
master_df = master_df.dropna(subset=['strain'])
print(f"After dropping missing strain: {len(master_df)}")

# Step 3: Remove hybrid strain
n_hybrid = (master_df['strain'] == HYBRID_STRAIN).sum()
master_df = master_df[master_df['strain'] != HYBRID_STRAIN]
print(f"Removed {n_hybrid} chunks from '{HYBRID_STRAIN}': {len(master_df)} remain")

# Step 4: Minimum chunk count per strain
strain_counts = master_df['strain'].value_counts()
valid_strains = strain_counts[strain_counts >= MIN_CHUNKS_PER_STRAIN].index
n_dropped = master_df['strain'].nunique() - len(valid_strains)
master_df = master_df[master_df['strain'].isin(valid_strains)]
print(f"Removed {n_dropped} rare strains (< {MIN_CHUNKS_PER_STRAIN} chunks): {len(master_df)} remain")

# Step 5: Sync numpy arrays
filtered_chunks = [all_chunks[i] for i in master_df['pos'].values]
master_df = master_df.reset_index(drop=True)

print(f"\nFinal dataset: {len(master_df)} chunks, "
      f"{master_df['strain'].nunique()} strains, "
      f"{master_df['animal_id'].nunique()} animals")
print(f"\nBreakdown by exp_stage × age:")
print(master_df.groupby(['age', 'exp_stage']).size().unstack(fill_value=0))

## 5. Frame-Level Data for GMM

In [ ]:
X_frames = np.vstack(filtered_chunks)
print(f"Frame matrix: {X_frames.shape}  ({len(filtered_chunks)} chunks × {CHUNK_FRAMES})")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_frames)

## 6. Subsample for Fitting

In [ ]:
if X_scaled.shape[0] > MAX_FIT_FRAMES:
    rng = np.random.RandomState(RANDOM_STATE)
    fit_idx = rng.choice(X_scaled.shape[0], MAX_FIT_FRAMES, replace=False)
    X_fit = X_scaled[fit_idx]
    print(f"Subsampled {MAX_FIT_FRAMES:,} from {X_scaled.shape[0]:,}")
else:
    X_fit = X_scaled
    print(f"Using all {X_scaled.shape[0]:,} frames")

## 7. GMM K-Sweep

In [ ]:
results = []
for k in K_VALUES:
    print(f"  K={k:>3d} ...", end="", flush=True)
    gmm = GaussianMixture(n_components=k, covariance_type=COVARIANCE_TYPE,
                           n_init=N_INIT, max_iter=MAX_ITER, random_state=RANDOM_STATE)
    gmm.fit(X_fit)
    bic, aic = gmm.bic(X_scaled), gmm.aic(X_scaled)
    ll = gmm.score(X_scaled) * X_scaled.shape[0]
    results.append({"K": k, "BIC": bic, "AIC": aic, "LL": ll,
                     "iters": gmm.n_iter_, "conv": gmm.converged_})
    print(f"  BIC={bic:,.0f}  AIC={aic:,.0f}  conv={gmm.converged_}")

results_df = pd.DataFrame(results)
print("K-sweep complete.")

## 8. BIC / AIC Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

best_bic_k = int(results_df.loc[results_df['BIC'].idxmin(), 'K'])
best_aic_k = int(results_df.loc[results_df['AIC'].idxmin(), 'K'])

for ax, col, color, best_k, label in [
    (axes[0], 'BIC', 'steelblue', best_bic_k, 'BIC (lower=better)'),
    (axes[1], 'AIC', 'coral', best_aic_k, 'AIC (lower=better)'),
    (axes[2], 'LL', 'seagreen', None, 'Log-Likelihood (higher=better)'),
]:
    ax.plot(results_df['K'], results_df[col], 'o-', color=color, markersize=5)
    ax.set_xlabel("K"); ax.set_ylabel(label); ax.set_title(label)
    if best_k:
        ax.axvline(best_k, color='red', ls='--', alpha=0.7, label=f'Best K={best_k}')
        ax.legend()

plt.suptitle(f"GMM K-Sweep — {STAGE} (cov={COVARIANCE_TYPE})", fontsize=14, y=1.02)
plt.tight_layout(); plt.show()
print(f"\n  BIC optimal: K={best_bic_k}")
print(f"  AIC optimal: K={best_aic_k}")

## 9. Normalized BIC/AIC

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(results_df['K'], results_df['BIC'] - results_df['BIC'].min(), 'o-',
        label=f'BIC (best K={best_bic_k})', color='steelblue')
ax.plot(results_df['K'], results_df['AIC'] - results_df['AIC'].min(), 's--',
        label=f'AIC (best K={best_aic_k})', color='coral')
ax.set_xlabel("K"); ax.set_ylabel("Criterion − min"); ax.set_title("Normalized BIC & AIC")
ax.legend(); plt.tight_layout(); plt.show()

## 10. Refit Best Model

In [ ]:
CHOSEN_K = best_bic_k
print(f"Refitting K={CHOSEN_K}...")
gmm_best = GaussianMixture(n_components=CHOSEN_K, covariance_type=COVARIANCE_TYPE,
                             n_init=5, max_iter=500, random_state=RANDOM_STATE)
gmm_best.fit(X_fit)
labels = gmm_best.predict(X_scaled)
print(f"Converged: {gmm_best.converged_}, {gmm_best.n_iter_} iters, {len(labels):,} frames labeled")

unique_l, counts = np.unique(labels, return_counts=True)
freq_order = np.argsort(-counts)
fig, ax = plt.subplots(figsize=(max(12, CHOSEN_K*0.3), 5))
ax.bar(range(len(unique_l)), counts[freq_order], color='steelblue', alpha=0.8)
ax.set_xlabel("Cluster (by freq)"); ax.set_ylabel("Frames"); ax.set_title(f"Cluster Frequency (K={CHOSEN_K})")
ax.set_xticks(range(len(unique_l))); ax.set_xticklabels(unique_l[freq_order], fontsize=7, rotation=90)
plt.tight_layout(); plt.show()

## 11. Per-Chunk Entropy

In [ ]:
chunk_entropies = []
for i in range(X_scaled.shape[0] // CHUNK_FRAMES):
    cl = labels[i*CHUNK_FRAMES:(i+1)*CHUNK_FRAMES]
    _, c = np.unique(cl, return_counts=True)
    chunk_entropies.append(entropy(c/c.sum(), base=2))

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(chunk_entropies, bins=40, color='steelblue', alpha=0.7, edgecolor='white')
ax.axvline(np.mean(chunk_entropies), color='red', ls='--',
           label=f'Mean={np.mean(chunk_entropies):.2f} bits')
ax.set_xlabel("Entropy (bits)"); ax.set_ylabel("Chunks"); ax.set_title(f"Per-Chunk Entropy (K={CHOSEN_K})")
ax.legend(); plt.tight_layout(); plt.show()
print(f"Mean: {np.mean(chunk_entropies):.3f} / Max: {np.log2(CHOSEN_K):.3f}")

## 12. Summary

In [ ]:
print("=" * 72)
print(f"  GMM K-Sweep: {STAGE} | {CHUNK_FRAMES}/{GAP_FRAMES} | cov={COVARIANCE_TYPE}")
print("=" * 72)
s = results_df.copy()
for c in ['BIC','AIC','LL']: s[c] = s[c].map(lambda x: f"{x:,.0f}")
print(s.to_string(index=False))
print(f"\n  BIC → K={best_bic_k}   AIC → K={best_aic_k}")